WaveNet

In [10]:
import json
import numpy as np
import soundfile as sf

import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()

from wavenet import WaveNetModel, mu_law_decode

from IPython.display import Audio

In [11]:
SAMPLES = 16000
TEMPERATURE = 1.0
WAVENET_PARAMS = './wavenet_params_default.json'
WAV_OUT = "generated.wav"

In [12]:
def generate_audio(model_dir, wavenet_params):
    net = WaveNetModel(
        batch_size=1,
        dilations=wavenet_params['dilations'],
        filter_width=wavenet_params['filter_width'],
        residual_channels=wavenet_params['residual_channels'],
        dilation_channels=wavenet_params['dilation_channels'],
        quantization_channels=wavenet_params['quantization_channels'],
        skip_channels=wavenet_params['skip_channels'],
        use_biases=wavenet_params['use_biases'],
        scalar_input=wavenet_params['scalar_input'],
        initial_filter_width=wavenet_params['initial_filter_width'],
        global_condition_channels=None,
        global_condition_cardinality=None
    )

    samples = tf.placeholder(tf.int32)

    next_sample = net.predict_proba(samples)

    decode = mu_law_decode(
        samples,
        wavenet_params["quantization_channels"]
    )

    sess = tf.Session()

    variables_to_restore = {
        var.name[:-2]: var for var in tf.global_variables()
        if not ('state_buffer' in var.name or 'pointer' in var.name)
    }

    saver = tf.train.Saver(variables_to_restore)
    saver.restore(sess, model_dir)

    print("Model załadowany")

    quantization_channels = wavenet_params["quantization_channels"]

    waveform = (
        [quantization_channels // 2]
        * (net.receptive_field - 1)
    )

    waveform.append(
        np.random.randint(quantization_channels)
    )

    for step in range(SAMPLES):
        if len(waveform) > net.receptive_field:
            window = waveform[-net.receptive_field:]
        else:
            window = waveform
        outputs = [next_sample]

        prediction = sess.run(outputs, feed_dict={samples: window})[0]

        scaled_prediction = np.log(
            np.maximum(prediction, 1e-12)
        ) / TEMPERATURE
        scaled_prediction = (scaled_prediction -
                                np.logaddexp.reduce(scaled_prediction))
        scaled_prediction = np.exp(scaled_prediction)
        if TEMPERATURE == 1.0:
                np.testing.assert_allclose(
                        prediction, scaled_prediction, atol=1e-5,
                        err_msg='Prediction scaling at temperature=1.0 '
                                'is not working as intended.')

        sample = np.random.choice(
            np.arange(quantization_channels),
            p=scaled_prediction
        )

        waveform.append(sample)

    audio = sess.run(
        decode,
        feed_dict={samples: waveform}
    )

    return audio
    

In [17]:
with open('./wavenet_params.json', 'r') as config_file:
    wavenet_params = json.load(config_file)

In [18]:
tf.reset_default_graph()

In [19]:
model_dir = './modele/sample_rate/model8k/model.ckpt-9999'
wavenet_params["sample_rate"] = 8e3
audio = generate_audio(model_dir, wavenet_params)

INFO:tensorflow:Restoring parameters from ./modele/sample_rate/model8k/model.ckpt-9999
Model załadowany


In [20]:
Audio(
    audio,
    rate=wavenet_params["sample_rate"]
)

In [21]:
tf.reset_default_graph()

In [22]:
model_dir = './modele/sample_rate/model4k/model.ckpt-9999'
wavenet_params["sample_rate"] = 4e3
audio = generate_audio(model_dir, wavenet_params)

INFO:tensorflow:Restoring parameters from ./modele/sample_rate/model4k/model.ckpt-9999
Model załadowany


In [23]:
Audio(
    audio,
    rate=wavenet_params["sample_rate"]
)

In [9]:
tf.reset_default_graph()